# Notebook 07 — Lake Mendota EDA
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polar-bear-after-lunch/AareML/blob/main/notebooks/07_lake_eda.ipynb)

Exploratory data analysis of Lake Mendota surface data from LakeBeD-US (McAfee et al., 2025).
Mirrors notebook 01 structure for direct river-vs-lake comparison.

**Key questions:**
- How does Lake Mendota's DO range, seasonality, and coverage compare to gauge 2473?
- Are the temporal patterns similar enough to justify the same modelling framework?
- What are the key differences that might explain the 3.4× RMSE gap?

In [1]:
# ── Colab setup (auto-runs only in Google Colab) ──────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import os
    from pathlib import Path

    # ── 1. Clone repo ──────────────────────────────────────────────────────
    if not Path('AareML').exists():
        os.system('git clone https://github.com/polar-bear-after-lunch/AareML.git')
    if not str(Path.cwd()).endswith('AareML'):
        os.chdir('AareML')

    # ── 2. Install dependencies ────────────────────────────────────────────
    os.system('pip install -q -r requirements.txt')

    # ── 3. Mount Google Drive ─────────────────────────────────────────────
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_DATA = Path('/content/drive/MyDrive/AareML_data')
    LOCAL_DATA = Path('data')
    LOCAL_DATA.mkdir(exist_ok=True)

    # ── 4. CAMELS-CH-Chem (needed for river comparison in nb06/07) ─────────
    DRIVE_CAMELS = DRIVE_DATA / 'camels-ch-chem'
    LOCAL_CAMELS = LOCAL_DATA / 'camels-ch-chem'
    if DRIVE_CAMELS.exists() and not LOCAL_CAMELS.exists():
        os.system(f'ln -s {DRIVE_CAMELS} {LOCAL_CAMELS}')
        print('CAMELS-CH-Chem loaded from Google Drive.')

    # ── 5. LakeBeD-US Lake Mendota ────────────────────────────────────────
    DRIVE_LAKE   = DRIVE_DATA / 'lakebed-us'
    LOCAL_LAKE   = LOCAL_DATA / 'lakebed-us'
    SURFACE_CSV  = DRIVE_LAKE / 'ME_daily_surface.csv'

    if SURFACE_CSV.exists():
        if not LOCAL_LAKE.exists():
            os.system(f'ln -s {DRIVE_LAKE} {LOCAL_LAKE}')
        print('LakeBeD-US data loaded from Google Drive.')
    else:
        print('Downloading Lake Mendota data to Drive (~194 MB, one-time)...')
        DRIVE_LAKE.mkdir(parents=True, exist_ok=True)
        os.system(
            'wget -q --show-progress -O /tmp/mendota.parquet '
            '"https://huggingface.co/datasets/eco-kgml/LakeBeD-US-CSE'
            '/resolve/main/Data/HighFrequency/ME/ME_Mendota_2D.parquet"'
        )
        os.system(f'mv /tmp/mendota.parquet {DRIVE_LAKE}/ME_Mendota_2D.parquet')
        # Pre-process
        os.system('python download_data.py --lake')
        # Move result to Drive
        os.system(f'mv data/lakebed-us/ME_daily_surface.csv {DRIVE_LAKE}/')
        os.system(f'ln -s {DRIVE_LAKE} {LOCAL_LAKE}')
        print('LakeBeD-US data saved to Google Drive for future sessions.')

    print(f'Setup complete. Working directory: {os.getcwd()}')


In [2]:
# ── CPU thread optimisation ────────────────────────────────────────────────
# PyTorch LSTM on CPU is fastest with 4-6 threads — beyond that,
# thread synchronisation overhead outweighs the parallelism gains.
import os, multiprocessing
N_CPU = multiprocessing.cpu_count()
N_THREADS = min(N_CPU, 6)  # clamp to 6 — empirically optimal on macOS
os.environ['OMP_NUM_THREADS']      = str(N_THREADS)
os.environ['MKL_NUM_THREADS']      = str(N_THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(N_THREADS)
print(f'CPU cores: {N_CPU} logical | Using {N_THREADS} threads for PyTorch')


CPU cores: 6 logical | Using 6 threads for PyTorch


In [3]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 130})

# G-U3 fix: auto-detect repo root (works both locally and in Colab after os.chdir)
_repo_root  = Path('.') if Path('figures').exists() else Path('..')
FIGURES_DIR = _repo_root / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)
DATA_DIR    = _repo_root / 'data' / 'lakebed-us'

## 1. Load Data

In [4]:
# Load daily surface data
surface_csv = DATA_DIR / 'ME_daily_surface.csv'
if surface_csv.exists():
    lake = pd.read_csv(surface_csv, parse_dates=['date'], index_col='date')
    print(f'Loaded from CSV: {len(lake)} daily rows')
else:
    print('ME_daily_surface.csv not found. Run: python download_data.py --lake')
    print('Or run notebook 06 first which generates this file.')
    raise FileNotFoundError(str(surface_csv))

# Reindex to continuous daily frequency (fill gaps with NaN)
idx = pd.date_range(lake.index.min(), lake.index.max(), freq='D')
lake = lake.reindex(idx)

print(f'\nLake Mendota surface data (depth ≈ 1m)')
print(f'Date range : {lake.index.min().date()} → {lake.index.max().date()}')
print(f'Total days : {len(lake)}')
print(f'\nVariable coverage:')
for col in lake.columns:
    cov = lake[col].notna().mean()
    valid = lake[col].dropna()
    print(f'  {col:12s}: {cov:5.1%}  |  '
          f'range [{valid.min():.2f}, {valid.max():.2f}]  |  '
          f'mean {valid.mean():.2f}  |  std {valid.std():.2f}')

Loaded from CSV: 3511 daily rows

Lake Mendota surface data (depth ≈ 1m)
Date range : 2006-06-28 → 2023-11-19
Total days : 6354

Variable coverage:
  do          : 51.5%  |  range [4.00, 24.34]  |  mean 10.05  |  std 2.40
  temp        : 53.9%  |  range [-0.33, 29.57]  |  mean 18.47  |  std 6.19
  chla_rfu    : 53.4%  |  range [-0.20, 14472.50]  |  mean 1118.99  |  std 1812.15
  phyco       : 51.5%  |  range [-956.00, 13593.00]  |  mean 568.91  |  std 1043.55
  par         : 27.3%  |  range [0.00, 650.20]  |  mean 29.84  |  std 45.57


## 2. Full Time Series (mirroring Fig 2 from notebook 01)

In [5]:
fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
colors = ['#A84B2F', '#01696F', '#006494', '#7A39BB']
labels = ['Temperature (°C)', 'DO (mg/L)', 'Chlorophyll-a (RFU)', 'Phycocyanin']
cols   = ['temp', 'do', 'chla_rfu', 'phyco']

for ax, col, color, label in zip(axes, cols, colors, labels):
    ax.plot(lake.index, lake[col], color=color, linewidth=0.7, alpha=0.85)
    ax.set_ylabel(label, fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Year')
plt.suptitle('Lake Mendota — Surface Sensor Time Series (depth ≈ 1m, daily median)',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '07_mendota_full_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 07_mendota_full_timeseries.png')

Saved: 07_mendota_full_timeseries.png


/var/folders/x4/23ky894n5lbdwq1cjhl4n5gh0000gn/T/ipykernel_56652/2855802887.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Seasonal DO Cycle (mirroring Fig 3 from notebook 01)
Compare with gauge 2473: DO ranges 8–14 mg/L with summer minimum.

In [6]:
# Monthly boxplot of DO — same as notebook 01 Fig 3
lake_use = lake[lake.index >= '2006-01-01'].copy()
lake_use['month'] = lake_use.index.month
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# DO seasonal cycle
ax = axes[0]
do_by_month = [lake_use[lake_use['month']==m]['do'].dropna().values
               for m in range(1, 13)]
bp = ax.boxplot(do_by_month, labels=month_labels, patch_artist=True,
                medianprops=dict(color='white', linewidth=2))
for patch in bp['boxes']:
    patch.set_facecolor('#01696F')
    patch.set_alpha(0.7)
ax.set_ylabel('Daily Median DO (mg/L)')
ax.set_title('Lake Mendota — Seasonal DO Cycle')
ax.grid(True, alpha=0.3)

# Temperature seasonal cycle
ax = axes[1]
temp_by_month = [lake_use[lake_use['month']==m]['temp'].dropna().values
                 for m in range(1, 13)]
bp2 = ax.boxplot(temp_by_month, labels=month_labels, patch_artist=True,
                 medianprops=dict(color='white', linewidth=2))
for patch in bp2['boxes']:
    patch.set_facecolor('#A84B2F')
    patch.set_alpha(0.7)
ax.set_ylabel('Daily Median Temperature (°C)')
ax.set_title('Lake Mendota — Seasonal Temperature Cycle')
ax.grid(True, alpha=0.3)

plt.suptitle('Lake Mendota — Seasonal Cycles (2006–2023)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '07_mendota_seasonal_cycles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 07_mendota_seasonal_cycles.png')

# Print seasonal stats
print('\nMonthly DO medians (mg/L):')
print(lake_use.groupby('month')['do'].median().round(2).to_string())

/var/folders/x4/23ky894n5lbdwq1cjhl4n5gh0000gn/T/ipykernel_56652/1527868692.py:13: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(do_by_month, labels=month_labels, patch_artist=True,
/var/folders/x4/23ky894n5lbdwq1cjhl4n5gh0000gn/T/ipykernel_56652/1527868692.py:26: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp2 = ax.boxplot(temp_by_month, labels=month_labels, patch_artist=True,


Saved: 07_mendota_seasonal_cycles.png

Monthly DO medians (mg/L):
month
1       NaN
2       NaN
3     13.86
4     13.57
5     10.73
6     10.52
7     10.61
8      9.05
9      8.75
10     8.27
11     9.39
12     8.41


/var/folders/x4/23ky894n5lbdwq1cjhl4n5gh0000gn/T/ipykernel_56652/1527868692.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. DO Distribution Comparison: River vs Lake

In [7]:
# Load river gauge 2473 data for comparison
import sys; sys.path.insert(0, '..')
from src.data import load_gauge, preprocess

raw_river = load_gauge('2473')
river = preprocess(raw_river)
river_test = river[river.index >= '2017-01-01']  # test period

# Lake test period (same 2017 onwards)
lake_test = lake_use[lake_use.index >= '2017-01-01']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# DO distributions
ax = axes[0]
river_do = river_test['O2C_sensor'].dropna()
lake_do  = lake_test['do'].dropna()
ax.hist(river_do, bins=50, alpha=0.6, color='#01696F', label=f'River gauge 2473 (n={len(river_do):,})')
ax.hist(lake_do,  bins=50, alpha=0.6, color='#A84B2F', label=f'Lake Mendota (n={len(lake_do):,})')
ax.set_xlabel('DO (mg/L)')
ax.set_ylabel('Count')
ax.set_title('DO Distribution — Test Period (2017+)')
ax.legend()
ax.grid(True, alpha=0.3)

# Temperature distributions
ax = axes[1]
river_temp = river_test['temp_sensor'].dropna()
lake_temp  = lake_test['temp'].dropna()
ax.hist(river_temp, bins=50, alpha=0.6, color='#01696F', label=f'River gauge 2473 (n={len(river_temp):,})')
ax.hist(lake_temp,  bins=50, alpha=0.6, color='#A84B2F', label=f'Lake Mendota (n={len(lake_temp):,})')
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Count')
ax.set_title('Temperature Distribution — Test Period (2017+)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('River vs Lake — Variable Distributions', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '07_river_vs_lake_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('DO statistics comparison:')
print(f"{'':20s} {'River 2473':>14s}  {'Lake Mendota':>14s}")
print(f"{'Mean (mg/L)':20s} {river_do.mean():>14.3f}  {lake_do.mean():>14.3f}")
print(f"{'Std (mg/L)':20s} {river_do.std():>14.3f}  {lake_do.std():>14.3f}")
print(f"{'Min (mg/L)':20s} {river_do.min():>14.3f}  {lake_do.min():>14.3f}")
print(f"{'Max (mg/L)':20s} {river_do.max():>14.3f}  {lake_do.max():>14.3f}")
print(f"{'Coverage':20s} {river_test['O2C_sensor'].notna().mean():>14.1%}  {lake_test['do'].notna().mean():>14.1%}")

[data] load_gauge 2473: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.01, 'ec_sensor': 0.007, 'O2C_sensor': 0.012}
DO statistics comparison:
                         River 2473    Lake Mendota
Mean (mg/L)                  11.403           9.993
Std (mg/L)                    0.956           2.226
Min (mg/L)                    9.190           4.370
Max (mg/L)                   13.700          19.320
Coverage                      96.4%           58.9%


/var/folders/x4/23ky894n5lbdwq1cjhl4n5gh0000gn/T/ipykernel_56652/1519606597.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Autocorrelation Analysis
The SHAP analysis found the LSTM has an effective memory of 3–4 days.
Does the autocorrelation structure explain why rivers are more predictable?

In [8]:
from pandas.plotting import autocorrelation_plot

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# River DO autocorrelation
river_do_clean = river['O2C_sensor'].dropna()
lake_do_clean  = lake_use['do'].dropna()

for row, (series, name, color) in enumerate([
    (river_do_clean, 'River Gauge 2473', '#01696F'),
    (lake_do_clean,  'Lake Mendota',     '#A84B2F'),
]):
    # ACF plot (manual, up to 30 lags)
    ax = axes[row, 0]
    lags = range(1, 31)
    acf_vals = [series.autocorr(lag=lag) for lag in lags]
    ax.bar(lags, acf_vals, color=color, alpha=0.7)
    ax.axhline(0, color='black', linewidth=0.8)
    # FIX NB07-2: Use non-NaN count for CI; np.sqrt makes intent explicit
    n_obs = series.count()  # non-NaN count
    ci = 1.96 / np.sqrt(n_obs)
    ax.axhline( ci, color='gray', linestyle='--', linewidth=0.8, label='95% CI')
    ax.axhline(-ci, color='gray', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Lag (days, 1 = yesterday)')
    ax.set_ylabel('Autocorrelation')
    ax.set_title(f'{name} — DO Autocorrelation\n(higher = more predictable at that lag distance)')
    ax.grid(True, alpha=0.3)

    # Rolling std (variability)
    ax = axes[row, 1]
    roll_std = series.rolling(30).std()
    ax.plot(series.index, roll_std, color=color, linewidth=0.8, alpha=0.8)
    ax.set_xlabel('Year')
    ax.set_ylabel('30-day Rolling Std (mg/L)')
    ax.set_title(f'{name} — DO Variability Over Time')
    ax.grid(True, alpha=0.3)

plt.suptitle('River vs Lake — DO Autocorrelation and Variability', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '07_river_vs_lake_autocorrelation.png', dpi=150, bbox_inches='tight')
plt.show()

print('DO autocorrelation at key lags:')
print(f"{'Lag':>5s}  {'River 2473':>12s}  {'Lake Mendota':>12s}")
for lag in [1, 2, 3, 7, 14, 21]:
    r_acf = river_do_clean.autocorr(lag=lag)
    l_acf = lake_do_clean.autocorr(lag=lag)
    print(f"{lag:>5d}  {r_acf:>12.3f}  {l_acf:>12.3f}")

DO autocorrelation at key lags:
  Lag    River 2473  Lake Mendota
    1         0.977         0.928
    2         0.948         0.877
    3         0.928         0.825
    7         0.898         0.662
   14         0.853         0.462
   21         0.808         0.332


/var/folders/x4/23ky894n5lbdwq1cjhl4n5gh0000gn/T/ipykernel_56652/741821218.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Data Coverage Comparison

In [9]:
# Year-by-year coverage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (series, name, color) in zip(axes, [
    (river['O2C_sensor'], 'River Gauge 2473', '#01696F'),
    (lake_use['do'],      'Lake Mendota',     '#A84B2F'),
]):
    yearly_cov = series.groupby(series.index.year).apply(lambda x: x.notna().mean())
    ax.bar(yearly_cov.index, yearly_cov.values, color=color, alpha=0.75)
    ax.axhline(0.1, color='red', linestyle='--', linewidth=1, label='10% threshold')
    ax.set_xlabel('Year')
    ax.set_ylabel('DO Coverage')
    ax.set_title(f'{name} — Annual DO Coverage')
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('River vs Lake — Annual DO Data Coverage', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '07_coverage_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/x4/23ky894n5lbdwq1cjhl4n5gh0000gn/T/ipykernel_56652/1268416337.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Summary: Does it make sense to compare?

In [10]:
# Print full comparison table
print('='*70)
print('RIVER vs LAKE COMPARISON SUMMARY')
print('='*70)

comparisons = {
    'Dataset':          ('CAMELS-CH-Chem',      'LakeBeD-US (McAfee 2025)'),
    'Location':         ('Switzerland (Alpine)', 'Wisconsin, USA (temperate)'),
    'Water body':       ('River (lotic)',        'Lake (lentic)'),
    'Depth':            ('Well-mixed, N/A',      '~1 m surface'),
    'Period':           ('1981–2020',            '2006–2023'),
    'DO coverage':      ('~97% (gauge 2473)',    '~51% (Mendota)'),
    'DO mean':          (f'{river_do.mean():.2f} mg/L', f'{lake_do.mean():.2f} mg/L'),
    'DO std':           (f'{river_do.std():.2f} mg/L',  f'{lake_do.std():.2f} mg/L'),
    'DO range':         (f'{river_do.min():.1f}–{river_do.max():.1f} mg/L',
                         f'{lake_do.min():.1f}–{lake_do.max():.1f} mg/L'),
    'Temp range':       (f'{river_temp.min():.1f}–{river_temp.max():.1f} °C',
                         f'{lake_temp.min():.1f}–{lake_temp.max():.1f} °C'),
    'DO autocorr(1d)':  (f'{river_do_clean.autocorr(1):.3f}',
                         f'{lake_do_clean.autocorr(1):.3f}'),
    'DO autocorr(7d)':  (f'{river_do_clean.autocorr(7):.3f}',
                         f'{lake_do_clean.autocorr(7):.3f}'),
    'Ridge DO RMSE':    ('0.303 mg/L',           '1.030 mg/L'),
    'LSTM DO RMSE':     ('0.299 mg/L',           '— (pending)'),
}

print(f"{'Property':25s}  {'River Gauge 2473':25s}  {'Lake Mendota':25s}")
print('-'*78)
for k, (v1, v2) in comparisons.items():
    print(f'{k:25s}  {v1:25s}  {v2:25s}')

print()
print('Conclusion: The comparison is scientifically meaningful but not controlled.')
print('Both systems show clear seasonal DO cycles and respond to temperature,')
print('justifying the same LSTM architecture. Key differences:')
print('  1. River DO is more autocorrelated (higher day-1 ACF) → more predictable')
print('  2. Lake DO has wider range → harder to constrain forecasts')
print('  3. Lake data is sparser (51% vs 97%) → more imputation noise')
print('  4. Different input features (lake: chla/phyco vs river: pH/EC)')

RIVER vs LAKE COMPARISON SUMMARY
Property                   River Gauge 2473           Lake Mendota             
------------------------------------------------------------------------------
Dataset                    CAMELS-CH-Chem             LakeBeD-US (McAfee 2025) 
Location                   Switzerland (Alpine)       Wisconsin, USA (temperate)
Water body                 River (lotic)              Lake (lentic)            
Depth                      Well-mixed, N/A            ~1 m surface             
Period                     1981–2020                  2006–2023                
DO coverage                ~97% (gauge 2473)          ~51% (Mendota)           
DO mean                    11.40 mg/L                 9.99 mg/L                
DO std                     0.96 mg/L                  2.23 mg/L                
DO range                   9.2–13.7 mg/L              4.4–19.3 mg/L            
Temp range                 1.1–18.3 °C                3.2–29.6 °C              
DO auto